# Пороги, калибровка и цена ошибки

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Выбирать порог по цене ошибок и проверять соответствие вероятностей частотам.

## От вероятности к действию

Пусть модель оценила шанс класса 1 как p. Порог 0.5 удобен, но не обязателен. Если пропустить положительный объект в десять раз дороже ложной тревоги, решение должно учитывать эту асимметрию. При стоимости FP равной c_FP и FN равной c_FN ожидаемая цена действия «1» равна c_FP(1−p), действия «0» — c_FN p. Выбираем 1, когда p≥c_FP/(c_FP+c_FN), если сами вероятности надёжны и других затрат нет.

Условия формулы важны: цены ошибок неотрицательны, $c_{FP}+c_{FN}>0$, цена правильного решения здесь принята равной нулю, p относится к положительному классу. При равенстве ожидаемых цен оба решения допустимы; правило выше выбирает 1. Если обе ошибки бесплатны, делить на их нулевую сумму нельзя, и по этому критерию любое решение одинаково.

## Разобранный пример: порог

При c_FP=1, c_FN=4 порог равен 1/5=0.2. Для p=0.3 тревога стоит в среднем 0.7, отсутствие тревоги — 1.2. Выбор 1 дешевле, хотя p меньше 0.5. Это вычисление не говорит, что каждый такой объект положителен: оно сравнивает средние последствия решения.

## Калибровка

Модель калибрована, если среди множества объектов с прогнозом около 0.8 положительный класс действительно встречается примерно в 80%. Это проверяют по группам вероятностей на данных, не использованных для обучения этих прогнозов. Одного объекта для проверки частоты недостаточно.

Модель может хорошо упорядочивать объекты, но быть чрезмерно уверенной. Калибровка изменяет вероятностную шкалу; выбор порога переводит шкалу в действия. Это разные шаги. И то и другое подбирают на данных разработки, а не многократным просмотром итогового теста.

В ноутбуке разбираем готовые авторские вероятности и цены. Это анализ решений, не результат обученной здесь модели. Отдельно сравниваем средний прогноз и фактическую долю в одной маленькой группе, не объявляя шесть объектов достаточным доказательством калибровки.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import numpy as np
p = np.array([0.1, 0.3, 0.8])
c_fp, c_fn = 1., 4.
threshold = c_fp / (c_fp + c_fn)
decisions = (p >= threshold).astype(int)
assert decisions.tolist() == [0, 1, 1]
positive_cost = c_fp * (1 - p)
negative_cost = c_fn * p
assert np.all(positive_cost[decisions == 1] <= negative_cost[decisions == 1])
print(threshold, decisions)  # 0.2 [0 1 1]


Теперь рассмотрим шесть авторских прогнозов около 0.8 с отдельно заданными ответами. Это другая таблица, а не продолжение массива `p`. `mean()` логических значений считает долю `True`.

In [ ]:
group_p = np.array([0.75, 0.78, 0.80, 0.80, 0.82, 0.85])
group_y = np.array([1, 1, 1, 1, 0, 0])
assert group_p.shape == group_y.shape
mean_probability = group_p.mean()
observed_fraction = (group_y == 1).mean()
assert np.isclose(mean_probability, 0.8)
assert np.isclose(observed_fraction, 4 / 6)
print("Средний прогноз:", round(mean_probability, 3))  # Средний прогноз: 0.8
print("Доля класса 1:", round(observed_fraction, 3))   # Доля класса 1: 0.667


В этой маленькой группе прогноз 0.8 сравнивается с частотой 4/6. Различие само по себе ещё не доказывает плохую калибровку: шести наблюдений мало для точного вывода о частоте в новых данных.

## Типичные ошибки

Не настраивайте порог по итоговому тесту. Не называйте изменение порога калибровкой. Проверяйте доступность реальных цен ошибок.

## Задачи

### 1. Базовая

Обязан ли порог равняться 0.5?

<details>
<summary>Подсказка</summary>

Цена ошибок может различаться.

</details>



### 2. Базовая

Что означает калибровка?

<details>
<summary>Подсказка</summary>

Сравните прогнозы группы с частотой ответов.

</details>



### 3. Прикладная

При c_FP=2,c_FN=3 найдите теоретический порог.

<details>
<summary>Подсказка</summary>

Разделите 2 на 2+3.

</details>



### 4. Прикладная

При p=0.3,c_FP=1,c_FN=4 сравните два действия.

<details>
<summary>Подсказка</summary>

Цены 1−p и 4p.

</details>



### 5. Олимпиадная

Почему точный расчёт цены не спасёт плохую вероятностную оценку?

<details>
<summary>Подсказка</summary>

Подставленные p могут быть неверны.

</details>



### 6. Олимпиадная

Можно ли доказать калибровку по одному прогнозу p=0.8 и метке 1?

<details>
<summary>Подсказка</summary>

Частота требует серии объектов.

</details>



## Итоги и самопроверка

Выведите порог из сравнения ожидаемых цен. Чем калибровка отличается от способности правильно ранжировать объекты?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Нет. Он зависит от цели и ограничений задачи.

### Задача 2

Соответствие заявленных вероятностей наблюдаемым долям в подходящих группах новых объектов.

### Задача 3

0.4 при указанных предположениях о ценах и вероятностях.

### Задача 4

0.7 против 1.2: выбрать положительный класс.

### Задача 5

Порог оптимален относительно предполагаемой вероятности, а не гарантированно относительно истинной.

### Задача 6

Нет. Один исход совместим со многими вероятностями, нужна достаточная группа и оценка неопределённости.